<div align="center">
    <br>
    <img alt="logo" src="https://avatars.githubusercontent.com/u/194089448?s=200&v=4" width="100"/>
    <p>
    Build a RAG Chatbot in Minutes
    </p>
    <p align="center">
       <a href="https://principled-intelligence.com/">Blog Article</a>&nbsp&nbsp | &nbsp&nbsp <a href="https://principled-intelligence.com">Principled Intelligence
        </a>
    </p>
    <hr/>
</div>

## Build a RAG Chatbot in Minutes

This notebook guides you through setting up a RAG chatbot over any document collection. Pick a model, pick a knowledge base, and you're done.

### What you will have at the end

- A RAG chatbot that answers questions over a document collection of your choice.
- A public URL you can share or point any testing tool at.
- A reproducible setup you can tweak. Swap the docs, the model, or the system prompt.


## What's in this notebook

This notebook spins up a **Retrieval-Augmented Generation (RAG) chatbot** in minutes. No infrastructure to manage, no complex setup.

You'll load a set of documents into a local knowledge base, start a chat server powered by an LLM of your choice, and get a public URL you can share or test with any tool.

The chatbot is powered by a **LLM** (GPT-5, Claude, Gemini, or anything else [litellm](https://docs.litellm.ai/) supports).


<img src="https://raw.githubusercontent.com/Principled-Intelligence/spectral-examples/main/assets/example-rag.svg" width="800"/>



test.svg

<details>
<summary><strong>🔍 How does it work?</strong></summary>

The RAG pipeline has four stages:

1. **Ingest** - your documents are split into chunks and converted into embeddings.
2. **Retrieve** - when a user asks a question, the closest chunks are fetched from the vector store.
3. **Augment** - those chunks are added to the LLM prompt as context.
4. **Generate** - the LLM produces an answer grounded in your documents.

ngrok provides the public tunnel so the chatbot, which runs locally in Colab, can be reached from anywhere.

</details>

## What you need

All credentials are loaded from **Colab Secrets** - click the 🔑 icon in the left sidebar, then add each key listed below. Nothing is hardcoded.

| Secret | Required | What it's for |
|---|---|---|
| `OPENAI_API_KEY` | **Yes** | Text embeddings for the knowledge base. |
| `ANTHROPIC_API_KEY` | Optional | Needed only if you switch the chat model to Claude. |
| `GOOGLE_API_KEY` | Optional | Needed only if you switch the chat model to Gemini. |
| `NGROK_TOKEN` | **Yes** | Opens the public tunnel. Get yours free at [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken). |

<details>
<summary>🔍 Why is an OpenAI key always required?</summary>
Text embeddings, the step that converts your documents into searchable vectors, always run through OpenAI's `text-embedding-3-small` model in this notebook. If you'd like to use a different embedding provider, you can change `EMBEDDING_MODEL` in the configuration cell.

</details>


### Step 1 - Add your API keys

Paste your API keys into Colab Secrets (🔑 icon in the left sidebar). This cell reads them into environment variables.


In [ ]:
from google.colab import userdata
import os

# OPENAI_API_KEY is mandatory for embedding generation
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

optional_llm_api_keys = ["ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]
for key in optional_llm_api_keys:
    try:
        os.environ[key] = userdata.get(key)
    except userdata.SecretNotFoundError:
        pass


#ngrok is needed to open the public tunnel.
ngrok_token = userdata.get("NGROK_TOKEN")


### Step 2 - Setup

This cell installs everything the notebook needs. The first run takes around 2 minutes.

<details>
<summary>🔍 What gets installed?</summary>

- `simple-chatbot` - the RAG server that indexes your documents and exposes a chat API.
- `pyngrok` - opens the public tunnel so your chatbot can be reached from anywhere.
- `datasets` - streams documents from HuggingFace without downloading the full dataset.
- Pinned `numpy` / `pyarrow` versions to avoid a known compatibility issue with `datasets`.

</details>


In [ ]:
import os

!pip install --force-reinstall -q "numpy>=2.0,<2.1" "pyarrow>=15" "datasets>=2.20" \
  git+https://github.com/Principled-Intelligence/simple-chatbot.git \
  pyngrok requests python-dotenv numpy > /dev/null 2>&1



### Step 3 - Configure your chatbot

Choose your knowledge base, model, and system prompt. The defaults work out of the box, change only what you want.

<details>
<summary>🔍 Swapping knowledge base, model, or prompt</summary>

**Swap the knowledge base:** point `HF_DATASET` at any HuggingFace dataset with a text column, then update `TEXT_FIELD` / `TITLE_FIELD` to match its schema. Set `HF_CONFIG` to `None` if your dataset doesn't have a configuration sub-split.

**Swap the chat model:** set `CHAT_MODEL` to any litellm-compatible string (e.g. `anthropic/claude-sonnet-4-6`, `google/gemini-2.0-flash`, `openai/gpt-5`). The corresponding API key must be in Colab Secrets and loaded in Step 1.

**Swap the system prompt:** rewriting `SYSTEM_PROMPT` changes the chatbot's persona and the rules it follows.

</details>


In [ ]:
# @title ⚙️ Chatbot configuration
HF_DATASET    = "wikimedia/wikipedia"  # @param {type:"string"}
KB_SIZE       = 300  # @param {type:"slider", min:50, max:1000, step:50}
CHAT_MODEL    = "openai/gpt-5.4-nano"  # @param ["openai/gpt-5.4-nano", "anthropic/claude-sonnet-4-6", "google/gemini-3.0-flash"]
SYSTEM_PROMPT = "You are a Wikipedia expert that replies to user queries."  # @param {type:"string"}

# ── advanced - rarely needs changing ──────────────────────────────────────────
HF_CONFIG       = "20231101.en"   # required for wikimedia/wikipedia, None otherwise
TEXT_FIELD      = "text"          # column containing the document body
TITLE_FIELD     = "title"         # column used for filenames; None to use row index
HF_SPLIT        = "train"
EMBEDDING_MODEL = "openai/text-embedding-3-small"
PORT            = 8000


### Step 4 - Create your knowledge base

Download the documents your chatbot will know about and save them locally.

<details>
<summary>🔍 More details</summary>

Streams `KB_SIZE` articles from HuggingFace and writes each one as a `.txt` file under `kbs/<dataset-name>/`. Streaming avoids a full dataset download - we only fetch the documents we need.

`simple-chatbot` reads this folder at startup via `--docs-dir`. Supported file types: `.txt`, `.md`, `.pdf`, `.docx`.

</details>


In [ ]:
from datasets import load_dataset

HF_NAME = HF_DATASET.split("/")[-1]
KB_DIR = f"kbs/{HF_NAME}"
import os
os.makedirs(KB_DIR, exist_ok=True)

print(f"Downloading first {KB_SIZE} documents from {HF_DATASET} (split={HF_SPLIT})...")

load_kwargs = dict(split=HF_SPLIT, streaming=True)
if HF_SPLIT is not None:
    dataset = load_dataset(HF_DATASET, HF_CONFIG, **load_kwargs)
else:
    dataset = load_dataset(HF_DATASET, **load_kwargs)

written = 0
for idx, row in enumerate(dataset):
    if written >= KB_SIZE:
        break
    # Filename: use TITLE_FIELD if set, else fall back to row index
    filename = str(row[TITLE_FIELD]).replace("/", "_").replace("\\", "_")[:100] if TITLE_FIELD is not None else f"doc_{idx}"
    body = row[TEXT_FIELD]
    with open(f"{KB_DIR}/{filename}.txt", "w") as f:
        f.write(body)
    written += 1

print(f"✓ Wrote {written} documents to {KB_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

✓ Wrote 300 documents to kbs/wikipedia


### Step 5 - Start the RAG chatbot server

Start the server in the background. The following script will index your documents and then expose a chat API.

<details>
<summary>🔍 What happens under the hood?</summary>

1. Scans `kbs/<dataset-name>/` for documents.
2. Chunks them (default: 500 tokens, 50-token overlap).
3. Embeds every chunk via OpenAI's `text-embedding-3-small`.
4. Stores the vectors in a local ChromaDB collection.
5. Binds a FastAPI server to `localhost:8000` exposing `POST /v1/chat/completions`.

Logs go to `/tmp/chatbot.log`.

</details>


In [ ]:
import time

!nohup simple-chatbot serve \
  --docs-dir {KB_DIR} \
  --chat-model "{CHAT_MODEL}" \
  --system-prompt "{SYSTEM_PROMPT}" \
  --embedding-model "{EMBEDDING_MODEL}" \
  --port {PORT} \
  --log-level INFO \
  > /tmp/chatbot.log 2>&1 &

print("Starting server... (wait for indexing)")

last = ""
while True:
    log = open("/tmp/chatbot.log").read()
    if "Starting HTTP server" in log:
        print("✓ Ready!")
        break
    last_line = log.strip().splitlines()[-1] if log.strip() else ""
    if last_line != last:
        print(last_line)
        last = last_line
    time.sleep(3)


Starting server... (wait for indexing)
✓ Ready!


### Step 6 - Make your chatbot public

Colab runs in an isolated environment that is not publicly reachable. This step creates a public URL that routes traffic to your local chatbot so it can be reached from anywhere.

Copy the printed URL, you'll use it to run tests or share your chatbot.

<details>
<summary>🔍 Technical details</summary>

[ngrok](https://ngrok.com/) opens an HTTPS tunnel from a public URL to the local `PORT`. The URL is ephemeral: it changes every time the tunnel restarts. For persistent or private connections, consider [spectral-bridge](https://github.com/Principled-Intelligence/spectral-bridge) instead.

</details>


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(PORT)
PUBLIC_URL = f"{tunnel.public_url}/v1/chat/completions/"
print(f"\n✓ Public URL: {PUBLIC_URL}\n")


✓ Public URL: https://durably-husked-scholar.ngrok-free.dev/v1/chat/completions/



### Step 7 - Try it out!

Sends a test message to your chatbot. If you get an answer back, everything is working.


In [ ]:
import json
import requests


def send_message(user_message: str):
  print(f"👤 User: {user_message}")
  response = requests.post(
      PUBLIC_URL,
      json={
          "messages": [
              {"role": "user", "content": user_message}
          ]
      },
      timeout=30,
  )
  response.raise_for_status()

  result = response.json()
  assistant_message = result["choices"][0]["message"]["content"]
  print(f"🤖 Assistant: {assistant_message}")


user_message = "Can you check in your provided documents when was Alabama founded?"
send_message(user_message)

print("-"*100)

user_message = "Name three of Andy Warhol's most famous works"
send_message(user_message)

print("-"*100)

user_message = "Briefly summarize one of Hercule Poirot's cases as described on his Wikipedia page"
send_message(user_message)







👤 User: Can you check in your provided documents when was Alabama founded?
🤖 Assistant: In the provided document, **Alabama was admitted as a state on December 14, 1819** (it’s described as “admitted as the 22nd state on December 14, 1819”).
----------------------------------------------------------------------------------------------------
👤 User: Name three of Andy Warhol's most famous works
🤖 Assistant: Three of Andy Warhol’s most famous works are:

- **Campbell’s Soup Cans** (1962)  
- **Marilyn Diptych** (1962)  
- **100 Dollar Bills** (1962)
----------------------------------------------------------------------------------------------------
👤 User: Briefly summarize one of Hercule Poirot's cases as described on his Wikipedia page
🤖 Assistant: One of Hercule Poirot’s early cases described on his Wikipedia page is **_The Mysterious Affair at Styles_**. During World War I, Poirot meets his friend **Captain Arthur Hastings**, and Poirot then solves this first case, which becomes the 

### Step 8 - (Optional) Download your knowledge base

Downloads the documents your chatbot was built on. Useful if you want to keep them or use them elsewhere.


In [ ]:
'''import shutil
from google.colab import files

# Zip the kbs folder
shutil.make_archive('/content/kbs', 'zip', '/content', 'kbs')

# Download
files.download('/content/kbs.zip')
'''

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Step 9 - Clean up

Stops the chatbot and closes the public URL when you're done.


In [ ]:
'''!pkill -f "simple-chatbot serve"
ngrok.kill()
'''

'!pkill -f "simple-chatbot serve"\nngrok.kill()\n'

In [ ]:
<img src="https://raw.githubusercontent.com/Principled-Intelligence/spectral-examples/main/assets/example-rag-spectral.svg" width="800"/>


## Want to test it automatically? Use Spectral!

Your chatbot is live. **Spectral** can probe it at scale by simulating user inputs and scoring the results with no extra code on your side.

architecture.svg

1. Open the [Spectral dashboard](https://spectral.principled.app/).
2. Create a target and paste the public URL from Step 6 as its endpoint.
3. Upload the knowledge base you just built (optional, improves factuality scoring).
4. Define the principles you want the chatbot to respect.
5. Launch an evaluation.

Within a few minutes you'll have a scored report flagging where your chatbot falls short.

### Troubleshooting

- **`ngrok` refuses to start** - `NGROK_TOKEN` is missing or invalid. Generate a fresh one at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken), re-run Steps 1 and 6.
- **Embedding errors during Step 5** - `OPENAI_API_KEY` is missing, expired, or out of quota. Text embeddings always go through OpenAI regardless of which chat model you picked.
- **Smoke test returns 502 or an empty response** - the chatbot is still indexing. Check `/tmp/chatbot.log` (Step 5) and retry once it logs `Starting HTTP server`.
- **Public URL stopped working** - the Colab runtime was recycled. Re-run Step 6 and update the URL. For longer runs, use [spectral-bridge](https://github.com/Principled-Intelligence/spectral-bridge).
